# 30 — BPE, Unigram Tokenization, and Vocabulary Design

**Master syllabus coverage:** V2 4.5–4.7

## Why this matters

Subword design sets sequence lengths, model matrix sizes, multilingual coverage, and what patterns become atomic. A tokenizer is learned data infrastructure, not a harmless preprocessing detail.

## Learning objectives

- Train and apply a small deterministic BPE merge list.
- Explain unigram tokenization and find a best segmentation with dynamic programming.
- Compare coverage, sequence length, and vocabulary-size tradeoffs.
- State why tokenizer versioning is inseparable from model weights.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Subwords trade vocabulary size for sequence length

Byte Pair Encoding begins with small symbols (here characters for readability), counts
adjacent pairs, and repeatedly merges the most frequent pair. Frequent sequences become
single tokens while rare words remain decomposable. Production tokenizers typically use
byte-level bases to guarantee coverage.


In [ ]:
from collections import Counter

word_frequencies = {
    "funny": 5,
    "funnier": 2,
    "funniest": 2,
    "timing": 4,
    "timed": 2,
}
vocabulary = {tuple(word) + ("</w>",): frequency for word, frequency in word_frequencies.items()}

def pair_counts(vocab):
    counts = Counter()
    for symbols, frequency in vocab.items():
        counts.update({pair: frequency for pair in zip(symbols, symbols[1:])})
    return counts

print(pair_counts(vocabulary).most_common(8))


## 2. Learn and apply BPE merges

Merge order is part of the tokenizer artifact. Encoding applies learned merges in rank
order, not by greedily choosing any currently longest substring. Deterministic tie-breaking
is necessary for reproducible vocabulary training.


In [ ]:
def merge_pair(symbols, pair):
    merged, index = [], 0
    while index < len(symbols):
        if index + 1 < len(symbols) and (symbols[index], symbols[index + 1]) == pair:
            merged.append(symbols[index] + symbols[index + 1])
            index += 2
        else:
            merged.append(symbols[index])
            index += 1
    return tuple(merged)

merges = []
for _ in range(10):
    counts = pair_counts(vocabulary)
    best = sorted(counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
    merges.append(best)
    vocabulary = {merge_pair(symbols, best): frequency for symbols, frequency in vocabulary.items()}

print("merge rules:", merges)
print("learned word segmentations:", list(vocabulary))


In [ ]:
def bpe_encode_word(word: str, learned_merges) -> tuple[str, ...]:
    symbols = tuple(word) + ("</w>",)
    for pair in learned_merges:
        symbols = merge_pair(symbols, pair)
    return symbols

for word in ("funny", "funnier", "timing", "untimed"):
    print(word, "→", bpe_encode_word(word, merges))


## 3. Unigram language-model tokenization

Unigram tokenization starts with a large candidate vocabulary and assigns each token a
probability. Dynamic programming finds the segmentation with highest probability; low-
utility candidates are pruned iteratively. Unlike BPE's deterministic merge history,
unigram defines a probabilistic segmentation model and can sample segmentations during
training. The small Viterbi example below illustrates inference, not full vocabulary EM.


In [ ]:
import math

token_prob = {"f": .04, "u": .04, "n": .05, "y": .03,
              "fu": .08, "fun": .20, "funny": .25, "ny": .07}

def best_unigram_segmentation(text: str):
    best = [(-math.inf, []) for _ in range(len(text) + 1)]
    best[0] = (0.0, [])
    for end in range(1, len(text) + 1):
        for start in range(end):
            token = text[start:end]
            if token in token_prob and best[start][0] > -math.inf:
                score = best[start][0] + math.log(token_prob[token])
                if score > best[end][0]:
                    best[end] = (score, [*best[start][1], token])
    if best[-1][0] == -math.inf:
        raise ValueError("text is not covered by candidate vocabulary")
    return best[-1]

print("unigram best:", best_unigram_segmentation("funny"))


## 4. Vocabulary evaluation

Compare coverage, average tokens per character/word, vocabulary storage, training speed,
and downstream quality. Larger vocabularies shorten sequences but enlarge embedding and
LM-head matrices (`V × C`) and may give rare tokens poorly estimated embeddings. The
tokenizer must be frozen and versioned with model checkpoints.


In [ ]:
evaluation_words = list(word_frequencies) + ["unfunny", "retiming"]
lengths = [len(bpe_encode_word(word, merges)) for word in evaluation_words]
character_lengths = [len(word) + 1 for word in evaluation_words]  # include boundary marker
print("mean BPE symbols:", sum(lengths) / len(lengths))
print("mean character symbols:", sum(character_lengths) / len(character_lengths))
assert all(length <= chars for length, chars in zip(lengths, character_lengths))


## Failure modes to recognize

- **Different merge order:** the same vocabulary strings produce different tokenizations.
- **No base coverage:** unseen text cannot be encoded losslessly.
- **Vocabulary too large:** embeddings dominate parameters and rare tokens are weakly trained.
- **Tokenizer changed after training:** token IDs no longer address the intended embeddings.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Run 0, 5, 10, and 20 BPE merges and plot vocabulary size against corpus token count.
2. Add byte-level base symbols conceptually and explain how raw bytes would display and decode.
3. Change unigram token probabilities until `funny` receives three different best segmentations.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can train and apply a subword scheme, test its coverage, and justify vocabulary size with model and data costs.

**Next:** 31 — Special tokens, controls, and prompt formats.
